<a href="https://colab.research.google.com/github/esraa9895/thesis-code/blob/main/BFGS_modify_multi_Dimantion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Self-contained implementation of non-linear optimization algorithms multi_Dimantion:

- BFGS
- l-BFGS
- memory less BFGS
- SBFGS

In [ ]:
import time
import numpy as np

In [ ]:
#get accuracy for each method
'''
def accuracy(f):
    return np.absolute(1.0-f)
'''

def accuracy(f,n):
    x = np.concatenate(([-0.5], np.zeros(n-1)))
    f_accurate = fun(x)
    return  1- (np.absolute(f_accurate - f)) #41. EG2(CUTE)

#### Rosenbrock function

In [ ]:

def fun_milt_dim(x,N):
     for i in range(int(N/2)):
        fun_dim =+100 * (x[2*i+1] - x[2*i]**2)**2 + (1 - x[2*i])**2
     return fun_dim

def fun(x):
    #N=n
    return fun_milt_dim(x,n)

def grad_milt_dim(x,N):
    grad_dim=[]
    for i in range(int(N/2)):
       grad_dim.append(200*(x[2*i+1]-x[2*i]**2)*(-2*x[2*i]) +2*(x[2*i]-1))
       grad_dim.append( 200*(x[2*i+1]-x[2*i]**2))
    if N%2 ==1:
       grad_dim.append(200*(x[2*i+2]-x[2*i+1]**2)*(-2*x[2*i+1]) +2*(x[2*i+1]-1))
    return grad_dim

def grad(x):
    #N=n
    return grad_milt_dim(x,n)

In [ ]:
def hessian_milt_dim(x, N):
    H = np.zeros((N, N))

    for i in range(N // 2):
        xi = 2 * i
        xi1 = 2 * i + 1

        # Second derivatives for the pair (x_i, x_i+1)
        H[xi, xi] += 1200 * x[xi]**2 - 400 * x[xi1] + 2
        H[xi, xi1] += -400 * x[xi]
        H[xi1, xi] += -400 * x[xi]
        H[xi1, xi1] += 200

    if N % 2 == 1:
        i = N - 2
        H[i, i] += 1200 * x[i]**2 - 400 * x[i+1] + 2
        H[i, i+1] += -400 * x[i]
        H[i+1, i] += -400 * x[i]
        H[i+1, i+1] += 200

    return H

def hessian(x):
    return hessian_milt_dim(x, len(x))

#### EG2 (cute)

In [ ]:

# =========================
# Function
# =========================
def fun(x):
    n = len(x)
    total = 0.0

    for i in range(n-1):
        total += np.sin(x[0] + x[i]**2 - 1)

    total += 0.5 * np.sin(x[-1]**2)

    return total

# =========================
# Gradient
# =========================
def grad(x):
    n = len(x)
    g = np.zeros(n)

    # derivative w.r.t x1
    g[0] = 2*x[0]*np.cos(x[0] + x[0]**2 - 1)
    for i in range(1,n-1):
        g[0] += np.cos(x[0] + x[i]**2 - 1)

    # derivatives w.r.t x2 ... x(n-1)
    for j in range(1, n-1):
        g[j] = 2 * x[j] * np.cos(x[0] + x[j]**2 - 1)

    # derivative w.r.t xn
    g[-1] = x[-1] * np.cos(x[-1]**2)

    return g


# =========================
# Hessian
# =========================
def hess(x):
    n = len(x)
    H = np.zeros((n, n))

    # ===== H11 =====
    for i in range(n-1):
        u = x[0] + x[i]**2 - 1
        H[0, 0] += -np.sin(u)

    # ===== mixed terms H1j and Hj1 =====
    for j in range(1, n-1):
        u = x[0] + x[j]**2 - 1
        val = -2 * x[j] * np.sin(u)

        H[0, j] = val
        H[j, 0] = val

    # ===== diagonal terms Hjj =====
    for j in range(1, n-1):
        u = x[0] + x[j]**2 - 1

        H[j, j] = (2 * np.cos(u)- 4 * x[j]**2 * np.sin(u))

    # ===== last element Hnn =====
    xn = x[-1]
    H[-1, -1] = ( np.cos(xn**2) - 2 * xn**2 * np.sin(xn**2) )

    return H

In [ ]:

# line-search conditions
def wolfe(f, g, xk, alpha, pk):
  c1 = 1e-4
  return f(xk + alpha * pk) <= f(xk) + c1 * alpha * np.dot(g(xk), pk)


def strong_wolfe(f, g, xk, alpha, pk, c2):
  # typically, c2 = 0.9 when using Newton or quasi-Newton's method.
  #            c2 = 0.1 when using non-linear conjugate gradient method.
  c1 = 1e-4

  return f(xk + alpha * pk) <= f(xk) + c1 * alpha * np.dot(g(xk), pk) and abs(
      np.dot(g(xk + alpha * pk), pk)) <= c2 * abs(np.dot(g(xk), pk))


def gold_stein(f, g, xk, alpha, pk, c):
  return (f(xk) + (1 - c) * alpha * np.dot(g(xk), pk) <= f(xk + alpha * pk)
    ) and (f(xk + alpha * pk) <= f(xk) + c * alpha * np.dot(g(xk), pk))


# line-search step len
def step_length(f, g, xk, alpha, pk, c2):
  return interpolation(f, g,
                       lambda alpha: f(xk + alpha * pk),
                       lambda alpha: np.dot(g(xk + alpha * pk), pk),
                       alpha, c2,
                       lambda f, g, alpha, c2: strong_wolfe(f, g, xk, alpha, pk, c2))


def interpolation(f, g, f_alpha, g_alpha, alpha, c2, strong_wolfe_alpha, iters=20):
  # referred implementation here:
  # https://github.com/tamland/non-linear-optimization
  l = 0.0
  h = 1.0
  for i in range(iters):
    if strong_wolfe_alpha(f, g, alpha, c2):
      return alpha

    half = (l + h) / 2
    alpha = - g_alpha(l) * (h**2) / (2 * (f_alpha(h) - f_alpha(l) - g_alpha(l) * h))
    if alpha < l or alpha > h:
      alpha = half
    if g_alpha(alpha) > 0:
      h = alpha
    elif g_alpha(alpha) <= 0:
      l = alpha
  return alpha

### methods

In [ ]:
def SR1(f, g, x0, iterations, error):
  xk = x0
  c2 = 0.9
  I = np.identity(xk.size)
  Hk = I
  X=[]
  Y=[]
  for i in range(iterations):
    # compute search direction
    gk =  np.array(g(xk))
    pk = -Hk.dot(gk)
    if np.linalg.norm(gk) < error:
        break

    # obtain step length by line search
    alpha = step_length(f, g, xk, 1.0, pk, c2)

    # update x
    xk1 = xk + alpha * pk
    gk1 =  np.array(g(xk1))

    # define sk and yk for convenience
    sk = xk1 - xk
    yk = gk1 - gk

    # Hs = Hk @ sk
    # if np.dot(yk - Hs, sk) > 1e-8:  # Ensure denominator is not too small
    #     Hk1 = Hk + np.outer((yk - Hs), (yk - Hs)) / np.dot((yk - Hs), sk)


    # compute H_{k+1} by SR1 update
    s_Hy = (sk-(Hk@ yk ))
    if(s_Hy.dot(yk)==0):
        print('divition by zero is not allowed')
        print("Did not converge.")
        break
    rho_k = float(1.0 / s_Hy.dot(yk))

        #rho_k = 1
    Hk1 = Hk+ rho_k* (s_Hy.dot(s_Hy))

    X.append(xk[0]), Y.append(xk[1])

    Hk = Hk1
    xk = xk1
  return xk, i + 1

In [ ]:
def Ml_SR1(f, g, x0, iterations, error):
  xk = x0
  c2 = 0.9
  I = np.identity(xk.size)
  Hk = I
  X=[]
  Y=[]
  iterations = 9000
  for i in range(iterations):
    # compute search direction
    gk =  np.array(g(xk))
    pk = -Hk.dot(gk)
    if np.linalg.norm(gk) < error:
        break

    # obtain step length by line search
    alpha = step_length(f, g, xk, 1.0, pk, c2)

    # update x
    xk1 = xk + alpha * pk
    gk1 = np.array(g(xk1))

    # define sk and yk for convenience
    sk = xk1 - xk
    yk = gk1 - gk

    # Hs = Hk @ sk
    # if np.dot(yk - Hs, sk) > 1e-8:  # Ensure denominator is not too small
    #     Hk1 = Hk + np.outer((yk - Hs), (yk - Hs)) / np.dot((yk - Hs), sk)


    # compute H_{k+1} by SR1 update
    s_Hy = (sk- yk)
    if(s_Hy.dot(yk)==0):
        print('divition by zero is not allowed')
        print("Did not converge.")
        break
    rho_k = float(1.0 / s_Hy.dot(yk))

        #rho_k = 1
    Hk1 = I+ rho_k* (s_Hy.dot(s_Hy))

    X.append(xk[0]), Y.append(xk[1])

    Hk = Hk1
    xk = xk1

  return xk, i + 1

In [ ]:

def bfgs(f, g, x0, iterations, error):
  xk = x0
  c2 = 0.9
  I = np.identity(xk.size)
  Hk = I

  for i in range(iterations):
    # compute search direction
    gk =  np.array(g(xk))
    pk = -Hk.dot(gk)

    # if np.linalg.norm(gk) < error:
    #   break

    # obtain step length by line search
    alpha = step_length(f, g, xk, 1.0, pk, c2)

    # update x
    xk1 = xk + alpha * pk
    gk1 = np.array(g(xk1))

    # define sk and yk for convenience
    sk = xk1 - xk
    yk = gk1 - gk

    # compute H_{k+1} by BFGS update
    rho_k = float(1.0 / yk.dot(sk))

    Hk1 = (I - rho_k * np.outer(sk, yk)).dot(Hk).dot(I - \
           rho_k * np.outer(yk, sk)) + rho_k * np.outer(sk, sk)

    if np.linalg.norm(xk1 - xk) < error:
      xk = xk1
      break

    Hk = Hk1
    xk = xk1

  return xk, i + 1

In [ ]:

def memory_less_bfgs(f, g, x0, iterations, error):
  xk = x0
  c2 = 0.9
  I = np.identity(xk.size)
  Hk = I
  X=[]
  Y=[]
  for i in range(iterations):
    # compute search direction
    gk = np.array(g(xk))
    pk = -Hk.dot(gk)

    # obtain step length by line search
    alpha = step_length(f, g, xk, 1.0, pk, c2)

    # update x
    xk1 = xk + alpha * pk
    gk1 = np.array(g(xk1))
    X.append(xk[0]), Y.append(xk[1])
    # define sk and yk for convenience
    sk = xk1 - xk
    yk = gk1 - gk

    # compute H_{k+1} by BFGS update
    rho_k = float(1.0 / yk.dot(sk))

    Hk1 = (I - rho_k * np.outer(sk, yk)).dot(I).dot(I - \
            rho_k * np.outer(yk, sk)) + rho_k * np.outer(sk, sk)
    if np.linalg.norm(xk1 - xk) < error:
      xk = xk1
      break

    Hk = Hk1
    xk = xk1

  return xk, i + 1

In [ ]:
def l_bfgs(f, g, x0, iterations, error, m=10):
  xk = x0
  c2 = 0.9
  I = np.identity(xk.size)
  sks = []
  yks = []
  X=[]
  Y=[]
  def Hp(H0):
    m_t = len(sks)
    q = g(xk)
    a = np.zeros(m_t)
    b = np.zeros(m_t)
    for i in reversed(range(m_t)):
      s = sks[i]
      y = yks[i]
      rho_i = float(1.0 / y.T.dot(s))
      a[i] = rho_i * s.dot(q)
      q = q - a[i] * y

    r = H0.dot(q)

    for i in range(m_t):
      s = sks[i]
      y = yks[i]
      rho_i = float(1.0 / y.T.dot(s))
      b[i] = rho_i * y.dot(r)
      r = r + s * (a[i] - b[i])

    return r

  for i in range(iterations):
    # compute search direction
    gk = np.array(g(xk))
    pk = -Hp(I)

    # if np.linalg.norm(gk) < error:
    #   break
    # obtain step length by line search
    alpha = step_length(f, g, xk, 1.0, pk, c2)

    # update x
    xk1 = xk + alpha * pk
    gk1 = np.array(g(xk1))

    # define sk and yk for convenience
    sk = xk1 - xk
    yk = gk1 - gk

    sks.append(sk)
    yks.append(yk)
    if len(sks) > m:
      sks = sks[1:]
      yks = yks[1:]

    X.append(xk[0]), Y.append(xk[1])

    if np.linalg.norm(xk1 - xk) < error:
      xk = xk1
      break

    xk = xk1

  return xk, i + 1

In [ ]:
    def scaled_bfgs(f, g, x0, iterations, error):
        xk = x0
        c2 = 0.9
        I = np.identity(xk.size)
        Hk = I
        X,Y = [],[]

        for i in range(iterations):
        # compute search direction
            gk =  np.array(g(xk))
            pk = -Hk.dot(gk)

            if np.linalg.norm(gk) < error:
                break

            # obtain step length by line search
            alpha = step_length(f, g, xk, 1.0, pk, c2)

            # update x
            xk1 = xk + alpha * pk
            gk1 = np.array(g(xk1))

            # define sk and yk for convenience
            sk = xk1 - xk
            yk = gk1 - gk

            # compute H_{k+1} by BFGS update
            rho_k = (1.0 / yk.T.dot(sk))

            #Hk1 = (I - rho_k * np.outer(sk, yk)).dot(Hk).dot(I - \
            #       rho_k * np.outer(yk, sk)) + rho_k * np.outer(sk, sk)

            #delta = (rho_k*6* (f(xk)-f(xk1)+sk.T.dot(gk1)))-2  #BFGSB
            #delta = (rho_k*2* (f(xk)-f(xk1)+sk.T.dot(gk1)))    #BFGSY
            delta = ( yk.T.dot(sk))/ (yk.T.dot(yk)) #rho_k*(yk.T.dot(yk)) #BFGSC

            '''
            #BFGSN
            Beta=np.exp(-1/np.power(i,2))
            delta = (yk.T.dot(sk))/ ((yk.T.dot(yk))+Beta)

            if alpha <np.exp(-1/np.power(i,2)):
                alpha = np.exp(-1/np.power(i,2))
                delta = 1
            '''

            Hk1 = Hk -rho_k*( (np.inner(np.outer(sk,yk), Hk) )+ np.outer(np.inner(Hk,yk.T) , sk) )+\
                    ((1.0/delta)+rho_k*(yk.T.dot(Hk).dot(yk)))*(rho_k* np.outer(sk,sk))

            '''
            Hk1 = (I - rho_k * np.outer(sk, yk)).dot(Hk).dot(I - \
                    rho_k * np.outer(yk, sk)) + rho_k * np.outer(sk, sk)
            '''
            # if np.linalg.norm(xk1 - xk) < error:
            #   xk = xk1
            #   break

            X.append(xk[0]), Y.append(xk[1])
            Hk = Hk1
            xk = xk1
        return xk, i + 1


In [ ]:
def double_scaled_bfgs(f, g, x0, iterations, error):
    xk = x0
    c2 = 0.9
    I = np.identity(xk.size)
    Hk = I
    X,Y = [],[]
    for i in range(iterations):
    # compute search direction
        gk =  np.array(g(xk))
        pk = -Hk.dot(gk)

        if np.linalg.norm(gk) < error:
              break

        # obtain step length by line search
        alpha = step_length(f, g, xk, 1.0, pk, c2)

        # update x
        xk1 = xk + alpha * pk
        gk1 =  np.array(g(xk1))

        # define sk and yk for convenience
        sk = xk1 - xk
        yk = gk1 - gk

        # compute H_{k+1} by BFGS update
        rho_k = (1.0 / yk.T.dot(sk))

        #Hk1 = (I - rho_k * np.outer(sk, yk)).dot(Hk).dot(I - \
         #       rho_k * np.outer(yk, sk)) + rho_k * np.outer(sk, sk)

        delta = (yk.T.dot(sk))/ ((yk.T.dot(yk))+(sk.T.dot(gk1)))
        alpha =(2-(delta*(yk.T.dot(yk)/yk.T.dot(sk))))/1

        delta = 1.0/delta
        alpha = 1.0/alpha

        Hk1 = alpha*I - alpha* (rho_k*( np.outer(sk,yk)+ np.outer(yk,sk)))+\
                    (delta+ alpha*rho_k*(yk.dot(yk)))*(rho_k* np.outer(sk,sk))
        '''
        Hk1 = (I - rho_k * np.outer(sk, yk)).dot(Hk).dot(I - \
                rho_k * np.outer(yk, sk)) + rho_k * np.outer(sk, sk)
        '''
        # if np.linalg.norm(xk1 - xk) < error:
        #   xk = xk1
        #   break

        X.append(xk[0]), Y.append(xk[1])
        Hk = Hk1
        xk = xk1
    return xk, i + 1


In [ ]:
def gradient_descent(f, g, x0, iterations, error):
    xk = x0
    c2 = 0.9

    for i in range(iterations):
      # compute search direction
      gk =  np.array(g(xk))
      pk = -gk
      if np.linalg.norm(gk) < error:
         break

      # obtain step length by line search
      alpha = step_length(f, g, xk, 1.0, pk, c2)

      # update x
      xk1 = xk + alpha * pk
      # if np.linalg.norm(xk1 - xk) < error:
      #   xk = xk1
      #   break

      xk = xk1
    return xk, i + 1

In [ ]:
def newton(f, g ,h, x0, iterations, error):
    xk = x0
    c2 = 0.9
    X,Y=[],[]

    for i in range(iterations):
      # compute search direction
      gk =  np.array(g(xk))
      Hk =  np.array(h(xk))

      pk = -Hk.dot(gk)
      if np.linalg.norm(gk) < error:
          break

      # obtain step length by line search
      alpha = step_length(f, g, xk, 1.0, pk, c2)
      # update x
      xk1 = xk + alpha * pk

      X.append(xk[0]), Y.append(xk[1])
     # print('np.linalg.norm(xk1 - xk) ', np.linalg.norm(xk1 - xk), 'and error is ' , error)

      # if np.linalg.norm(xk1 - xk) < error:
      #    xk = xk1
      #    break

      Hk= h(xk1)
      xk = xk1
    return xk, i + 1

In [ ]:
# Residuals for multi-dimensional Rosenbrock function
def rosenbrock_residuals(x):
    n = len(x)
    residuals = []
    for i in range(0, n - 1, 2):
        r1 = 10 * (x[i + 1] - x[i] ** 2)
        r2 = 1 - x[i]
        residuals.extend([r1, r2])
    return np.array(residuals)

# Jacobian of the residuals for multi-dimensional Rosenbrock
def jacobian_rosenbrock(x):
    n = len(x)
    m = n if n % 2 == 0 else n - 1  # handle odd-length x (ignore last term)
    J = np.zeros((m, n))

    row = 0
    for i in range(0, m, 2):
        # r1 = 10 * (x[i+1] - x[i]**2)
        J[row, i] = -20 * x[i]
        J[row, i + 1] = 10
        row += 1
        # r2 = 1 - x[i]
        J[row, i] = -1
        row += 1
    return J


# Residuals for multi-dimensional EG2 function
def EG2_residuals(x):
    n = len(x)
    r = np.zeros(n)

    for i in range(n-1):
        r[i] = np.sin(x[0] + x[i]**2 - 1)

    r[n-1] = np.sqrt(0.5) * np.sin(x[n-1]**2)

    return r

In [ ]:
# Jacobian of the residuals for multi-dimensional EG2
def jacobian_EG2(x):
    n = len(x)
    J = np.zeros((n, n))

    # matrex 0_n-1
    for i in range(n-1):
        u = x[0] + x[i]**2 - 1

        # for x0
        J[i, 0] = np.cos(u)

        # for x1
        J[i, i] = 2 * x[i] * np.cos(u)

    # last column
    J[n-1, n-1] = np.sqrt(0.5) * 2 * x[n-1] * np.cos(x[n-1]**2)

    return J

# Gauss-Newton algorithm

def gauss_newton(f, g , x0, iterations, error):
    xk= x0
    X,Y= [],[]
    for i in range(iterations):
        # Calculate residuals and Jacobian
        residuals = EG2_residuals(xk)
        J = jacobian_EG2(xk)
        gk = np.array(g(xk))
        if np.linalg.norm(gk) < error:
            break

        # Gauss-Newton update direction
        pk = - np.linalg.inv(J.T @ J) @ J.T @ residuals

        # Wolfe line search to determine step size alpha_k
        alpha = step_length(f, g, xk,1.0, pk,c2= 0.9)

        # Update the parameter estimate
        xk1 = xk + alpha* pk
        X.append(xk[0]), Y.append(xk[1])

        xk = xk1
    return xk, i + 1

In [ ]:
def levenberg_marquardt(f, g, x0, iterations, error, mu=1e-3):
    """
    Levenberg-Marquardt algorithm for solving nonlinear least squares problems,
    applied to the Rosenbrock function expressed as a sum of squares.

    Parameters:
        f : callable
            Objective function to minimize, typically sum of squared residuals.
        g : callable
            Gradient of the objective function.
        x0 : ndarray
            Initial guess for the solution.
        iterations : int
            Maximum number of iterations to run.
        error : float
            Tolerance for convergence based on the gradient norm.
        mu : float, optional
            Initial damping parameter (default is 1e-3).

    Returns:
        xk : ndarray
            Estimated solution vector after convergence or max iterations.
        i + 1 : int
            Number of iterations performed.
    """
    xk = x0
    X, Y = [], []

    for i in range(iterations):
        # Compute residuals and Jacobian
        residuals = EG2_residuals(xk)
        J = jacobian_EG2(xk)
        gk =np.array(g(xk))

        # stopping condition based on gradient norm
        if np.linalg.norm(gk) < error:
            break

        # Hessian approximation and LM direction
        #LM step: (JᵀJ + μI) d = -Jᵀr
        H = J.T @ J
        A = H + mu * np.eye(len(xk))
        pk = -np.linalg.solve(A, J.T @ residuals)

        # Try step with current direction
        xk1 = xk + pk
        actual_reduction = np.linalg.norm(residuals)**2 - np.linalg.norm(EG2_residuals(xk1))**2

        # Update damping parameter mu
        # Accept step if reduction is sufficie
        if actual_reduction > 0:
            xk = xk1
            mu *= 0.8 # decrease damping
            X.append(xk[0]), Y.append(xk[1])
        else:
            mu *= 2  # increase damping if no improvement
    return xk, i + 1

### apply methods

In [ ]:
n = 10
x0 = np.array(np.ones(n))
error = 1e-5
max_iterations = 1000

In [ ]:

print('\n======= SR1 ======\n')
start = time.time()
x, n_iter = SR1(fun, grad, x0, max_iterations, error)
end = time.time()
print ("  SR1 terminated in {} iterations, x = {}, f(x) = {:.4e}, time elapsed {:.4f}, time per iter {:.4e}"
  .format(n_iter, x, fun(x), end - start, (end - start) / n_iter))
print('  SR1 accuracy', accuracy(fun(x),n))


======= SR1 ======

  SR1 terminated in 1000 iterations, x = [nan nan nan nan nan], f(x) = nan, time elapsed 2.5858, time per iter 2.5858e-03
  SR1 accuracy nan


In [ ]:

print('\n======= Ml_SR1 ======\n')
start = time.time()
x, n_iter = Ml_SR1(fun, grad, x0, max_iterations, error)
end = time.time()
print ("  Ml_SR1 terminated in {} iterations, x  f(x) = {:.4e}, time elapsed {:.4f}, time per iter {:.4e}"
  .format(n_iter,  fun(x), end - start, (end - start) / n_iter))
print('  Ml_SR1 accuracy', accuracy(fun(x),n))


======= Ml_SR1 ======



/tmp/ipython-input-2108366779.py:41: RuntimeWarning: invalid value encountered in scalar multiply
  Hk1 = I+ rho_k* (s_Hy.dot(s_Hy))


  Ml_SR1 terminated in 9000 iterations, x  f(x) = nan, time elapsed 23.6289, time per iter 2.6254e-03
  Ml_SR1 accuracy nan


In [ ]:
print( '\n======= Broyden-Fletcher-Goldfarb-Shanno ======\n')
start = time.time()
x, n_iter = bfgs(fun, grad, x0, max_iterations, error)
end = time.time()
print ("  BFGS terminated in {} iterations, x  f(x) = {:.4e}, time elapsed {:.4f}, time per iter {:.4e}"
  .format(n_iter, fun(x), end - start, (end - start) / n_iter))
print('  BFGS accuracy', accuracy(fun(x),n))


======= Broyden-Fletcher-Goldfarb-Shanno ======

  BFGS terminated in 10 iterations, x  f(x) = -3.9339e+00, time elapsed 0.0087, time per iter 8.7478e-04
  BFGS accuracy 0.9924213020071129


In [ ]:
1-((-8.94 +9.50)/10)

In [ ]:
print('\n======= Memory-less BFGS ======\n')
start = time.time()
x, n_iter = memory_less_bfgs(fun, grad, x0,
                             iterations=max_iterations, error=error)
end = time.time()
print("  Memory-less BFGS terminated in {} iterations, x , f(x) = {:.4e}, time elapsed {:.4f}, time per iter {:.4e}"
      .format(n_iter,  fun(x), end - start, (end - start) / n_iter))
print('  Memory-less BFGS accuracy:', accuracy(fun(x),n))


======= Memory-less BFGS ======

  Memory-less BFGS terminated in 25 iterations, x , f(x) = -9.0000e+00, time elapsed 0.0183, time per iter 7.3247e-04
  Memory-less BFGS accuracy: 0.9289445349555709


In [ ]:
print('\n======= Limited-memory BFGS (L-BFGS) ======\n')
start = time.time()
x, n_iter = l_bfgs(fun, grad, x0,
                   iterations=max_iterations, error=error)
end = time.time()
print("  L-BFGS terminated in {} iterations, x , f(x) = {:.4e}, time elapsed {:.4f}, time per iter {:.4e}"
      .format(n_iter,  fun(x), end - start, (end - start) / n_iter))
print('  L-BFGS accuracy:', accuracy(fun(x),n))


======= Limited-memory BFGS (L-BFGS) ======

  L-BFGS terminated in 43 iterations, x , f(x) = -9.2981e+00, time elapsed 0.1214, time per iter 2.8225e-03
  L-BFGS accuracy: 0.6308326256203909


In [ ]:
x = np.concatenate(([-0.5], np.zeros(n-1)))
fun(x)

np.float64(-18.903894378228568)

 10 Dimention

 * BFGS terminated in 14 iterations, x  f(x) = -9.5000e+00, time elapsed 0.0380, time per iter 2.7178e-03
  BFGS accuracy 0.4289445182944416

*   L-BFGS terminated in 56 iterations, x , f(x) = -6.7003e+00, time elapsed 0.3728, time per iter 6.6570e-03
  L-BFGS accuracy: -1.2286107094332133

*  ======= gauss_newton ======

 gauss_newton terminated in 1000 iterations, x  f(x) = -3.7953e+00, time elapsed 5.1328, time per iter 5.1328e-03
  gauss_newton accuracy -4.133671168669749
  *
======= levenberg_marquardt ======

  levenberg_marquardt terminated in 1000 iterations, x =  f(x) = 1.0004e-23, time elapsed 0.2140, time per iter 2.1404e-04
  levenberg_marquardt accuracy -7.928944512188021



In [ ]:
1- np.power((-9.5+8.94),2)

np.float64(0.6863999999999995)

In [ ]:
print('\n======= Scaled BFGS ======\n')
start = time.time()
x, n_iter = scaled_bfgs(fun, grad, x0, max_iterations, error)
end = time.time()
print ("  S-BFGS terminated in {} iterations, x, f(x) = {:.4e}, time elapsed {:.4f}, time per iter {:.4e}"
  .format(n_iter,  fun(x), end - start, (end - start) / n_iter))
print('  S-BFGS accuracy', accuracy(fun(x),n))


======= Scaled BFGS ======



/tmp/ipykernel_11412/1047550847.py:9: RuntimeWarning: overflow encountered in scalar power
  total += np.sin(x[0] + x[i]**2 - 1)
/tmp/ipykernel_11412/1047550847.py:9: RuntimeWarning: invalid value encountered in sin
  total += np.sin(x[0] + x[i]**2 - 1)


  S-BFGS terminated in 1000 iterations, x, f(x) = nan, time elapsed 4.6085, time per iter 4.6085e-03
  S-BFGS accuracy nan


In [ ]:

print('\n======= double_scaled_bfgs ======\n')
start = time.time()
x, n_iter = double_scaled_bfgs(fun, grad, x0, max_iterations, error)
end = time.time()
print ("  double_scaled_bfgs terminated in {} iterations, x  f(x) = {:.4e}, time elapsed {:.4f}, time per iter {:.4e}"
  .format(n_iter, fun(x), end - start, (end - start) / n_iter))
print('  double_scaled_bfgs accuracy', accuracy(fun(x),n))


======= double_scaled_bfgs ======



/tmp/ipython-input-715586532.py:33: RuntimeWarning: invalid value encountered in scalar multiply
  alpha =(2-(delta*(yk.T.dot(yk)/yk.T.dot(sk))))/1
/tmp/ipython-input-715586532.py:35: RuntimeWarning: divide by zero encountered in scalar divide
  delta = 1.0/delta


  double_scaled_bfgs terminated in 1000 iterations, x  f(x) = nan, time elapsed 3.5367, time per iter 3.5367e-03
  double_scaled_bfgs accuracy nan


In [ ]:
print('\n======= gradient_descent ======\n')
start = time.time()
x, n_iter = gradient_descent(fun, grad, x0, max_iterations, error)
end = time.time()
print ("  gradient_descent terminated in {} iterations, x , f(x) = {:.4e}, time elapsed {:.4f}, time per iter {:.4e}"
  .format(n_iter,  fun(x), end - start, (end - start) / n_iter))
print('  gradient_descent accuracy', accuracy(fun(x),n))


======= gradient_descent ======

  gradient_descent terminated in 69 iterations, x , f(x) = -3.9339e+00, time elapsed 0.1489, time per iter 2.1574e-03
  gradient_descent accuracy 0.9924210932499045


In [ ]:
print('\n======= newton ======\n')
start = time.time()
x, n_iter = newton(fun, grad,hess, x0, max_iterations, error)
end = time.time()
print ("  newton terminated in {} iterations, x =  f(x) = {:.4e}, time elapsed {:.4f}, time per iter {:.4e}"
  .format(n_iter,  fun(x), end - start, (end - start) / n_iter))
print('newton accuracy', accuracy(fun(x),n))


======= newton ======



/tmp/ipython-input-1550782360.py:9: RuntimeWarning: overflow encountered in scalar power
  total += np.sin(x[0] + x[i]**2 - 1)
/tmp/ipython-input-1550782360.py:9: RuntimeWarning: invalid value encountered in sin
  total += np.sin(x[0] + x[i]**2 - 1)


  newton terminated in 1000 iterations, x =  f(x) = nan, time elapsed 2.7197, time per iter 2.7197e-03
newton accuracy nan


In [ ]:
print('\n======= gauss_newton ======\n')
start = time.time()
x, n_iter = gauss_newton(fun, grad, x0, max_iterations, error)
end = time.time()
print (" gauss_newton terminated in {} iterations, x  f(x) = {:.4e}, time elapsed {:.4f}, time per iter {:.4e}"
  .format(n_iter,  fun(x), end - start, (end - start) / n_iter))
print('  gauss_newton accuracy', accuracy(fun(x),n))


======= gauss_newton ======

 gauss_newton terminated in 1000 iterations, x  f(x) = -3.7953e+00, time elapsed 4.2641, time per iter 4.2641e-03
  gauss_newton accuracy -4.133671168669752


In [ ]:
1- (-4.82 +3.94)

1.8800000000000003

In [ ]:
print('\n======= levenberg_marquardt ======\n')
start = time.time()
x, n_iter = levenberg_marquardt(fun, grad, x0, max_iterations, error)
end = time.time()
print ("  levenberg_marquardt terminated in {} iterations, x =  f(x) = {:.4e}, time elapsed {:.4f}, time per iter {:.4e}"
  .format(n_iter,  fun(x), end - start, (end - start) / n_iter))
print('  levenberg_marquardt accuracy', accuracy(fun(x),n))


======= levenberg_marquardt ======

  levenberg_marquardt terminated in 1000 iterations, x =  f(x) = 4.8986e-16, time elapsed 0.1530, time per iter 1.5301e-04
  levenberg_marquardt accuracy -2.9414695791677503
